In [1]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

import os
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

from story_llm.model import StoryModel
from story_llm.config import ModelConfig

from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.amp import autocast, GradScaler


import time

/home/mrudhuhas/Documents/Projects/llm-scratch/story-llm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT_DIR = Path().resolve().parent
DATA_DIR = ROOT_DIR / "data"
TOKENIZER_DIR = ROOT_DIR / "tokenizer"
TOKENIZER_NAME = "tiny_stories_tokenizer.json"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

train_size = 200000
test_size = 10000

torch.set_float32_matmul_precision("high")

In [3]:
tiny_stories_train = load_dataset("roneneldan/TinyStories", split="train", cache_dir=DATA_DIR)
tiny_stories_test = load_dataset("roneneldan/TinyStories", split="validation", cache_dir=DATA_DIR)

tiny_stories_train = tiny_stories_train.shuffle().select(range(train_size))
tiny_stories_test = tiny_stories_test.shuffle().select(range(test_size))

In [4]:
print(tiny_stories_train[5]['text'])

Once upon a time, there was a bright stamp. It was red, blue and yellow. The stamp had a big heart on it. The stamp was very happy.

One day, a little boy found the stamp. The boy loved the stamp very much. He put the stamp on a letter. The letter was for his best friend, Tim.

The stamp went on a long trip. It saw many things. At last, the letter came to Tim. Tim was very happy to see the bright stamp, too. He knew his friend loved him.


In [5]:
def build_tokenizer(texts, vocab_size=10000, min_frequency=2):
    tokenizer = Tokenizer(BPE())
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<s>", "<pad>", "</s>"],
        min_frequency=min_frequency,
    )

    tokenizer.train_from_iterator(texts, trainer=trainer)

    tokenizer.post_processor = TemplateProcessing(
        single="<s> $A </s>",
        special_tokens=[("<s>", tokenizer.token_to_id("<s>")), ("</s>", tokenizer.token_to_id("</s>"))],
    )

    tokenizer.decoder = ByteLevelDecoder()

    tokenizer.save(os.path.join(TOKENIZER_DIR, TOKENIZER_NAME))

if not os.path.exists(os.path.join(TOKENIZER_DIR, TOKENIZER_NAME)):
    build_tokenizer(tiny_stories_train['text'], vocab_size=10000, min_frequency=2)

tokenizer = Tokenizer.from_file(os.path.join(TOKENIZER_DIR, TOKENIZER_NAME))

In [6]:
def tokenize_and_concat(texts, tokenizer):
    all_token_ids = []
    for text in texts:
        encoded = tokenizer.encode(text)
        all_token_ids.extend(encoded.ids)
    return torch.tensor(all_token_ids, dtype=torch.long)

In [7]:
class TinyStoriesDataset(Dataset):
    def __init__(self, text: str, tokenizer, max_len, stride):
        self.max_len = max_len
        self.stride = stride
        self.data = tokenize_and_concat(text, tokenizer)

    def __len__(self):
        return (len(self.data) - self.max_len) // self.stride 
    
    def __getitem__(self, idx):
        start = idx * self.stride
        x = self.data[start : start + self.max_len]
        y = self.data[start + 1 : start + self.max_len + 1]
        return x, y


In [8]:
stride = 128
max_len = 512

train_dataset = TinyStoriesDataset(tiny_stories_train['text'], tokenizer, max_len, stride)
test_dataset = TinyStoriesDataset(tiny_stories_test['text'], tokenizer, max_len, stride)

In [9]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=4, prefetch_factor=2, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=4, prefetch_factor=2, persistent_workers=True)

In [10]:
sample_batch = next(iter(train_loader))

In [11]:
# decode first sample from the batch and print
first_input_ids, first_target_ids = sample_batch[0][2], sample_batch[1][2]
decoded_input = tokenizer.decode(first_input_ids.tolist())
decoded_target = tokenizer.decode(first_target_ids.tolist())

In [12]:
decoded_input

' they can decorate it. She says they can use some whipped cream and more nuts to make it look pretty.\n\nMia and her mom provide the cake with a nice topping. They also write "Happy Birthday Dad" on the card and stick the candles on the cake. They hide the cake in the fridge and wait for dad to come home. Mia is happy. She thinks dad will love the cake. She thinks it is the most unique cake ever. She hugs her mom and says thank you. Her mom hugs her back and says you\'re welcome. She says she loves her very much. Mia says she loves her too. They smile and wait for dad.Tom and Lily were curious twins who liked to explore the farm. One day, they saw a big machine that was used to fix the corn. It had wheels, pipes, and buttons.\n\n"Can we play with it?" Tom asked.\n\n"No, it is not a toy. It is for work. Dad said we should not touch it," Lily said.\n\nBut Tom was too curious. He wanted to see what the machine could do. He climbed on it and pushed a button. The machine made a loud noise 

In [13]:
decoded_target

' can decorate it. She says they can use some whipped cream and more nuts to make it look pretty.\n\nMia and her mom provide the cake with a nice topping. They also write "Happy Birthday Dad" on the card and stick the candles on the cake. They hide the cake in the fridge and wait for dad to come home. Mia is happy. She thinks dad will love the cake. She thinks it is the most unique cake ever. She hugs her mom and says thank you. Her mom hugs her back and says you\'re welcome. She says she loves her very much. Mia says she loves her too. They smile and wait for dad.Tom and Lily were curious twins who liked to explore the farm. One day, they saw a big machine that was used to fix the corn. It had wheels, pipes, and buttons.\n\n"Can we play with it?" Tom asked.\n\n"No, it is not a toy. It is for work. Dad said we should not touch it," Lily said.\n\nBut Tom was too curious. He wanted to see what the machine could do. He climbed on it and pushed a button. The machine made a loud noise and s

In [14]:
print(f"Train Tokens: {len(train_dataset.data)}")
print(f"Test Tokens: {len(test_dataset.data)}")
print(f"Total Train Samples: {len(train_dataset)}")
print(f"Total Test Samples: {len(test_dataset)}")
print(f"Vocab Size: {tokenizer.get_vocab_size()}")


Train Tokens: 44200908
Test Tokens: 2145030
Total Train Samples: 345315
Total Test Samples: 16754
Vocab Size: 8192


In [15]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


config = ModelConfig(
    vocab_size=tokenizer.get_vocab_size(),
    block_size=max_len,
    n_layers=6,
    n_heads=8,
    d_model=512,
    d_ffn=4*512,
    attn_dropout=0.1,
    ffn_dropout=0.1,
    bias=True,
    pre_norm=False,
    device=device,
    num_epochs=3,
    batch_size=batch_size,
    learning_rate=1e-4,
)
model = StoryModel(config=config).to(device)
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1,
    eps=1e-8,
    fused=torch.cuda.is_available()
)
criterion = nn.CrossEntropyLoss()

total_steps = config.num_epochs * len(train_loader)
warmup_steps = int(0.02 * total_steps)

warmup_scheduler = LinearLR(optimizer, start_factor=1e-8, end_factor=1.0, total_iters=warmup_steps)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=1e-5)

scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])

In [16]:
import wandb

wandb.login()
wandb.init(
    project="story-llm",
    name="E1-baseline-postln",
    config={**config.__dict__, "total_steps": total_steps, "warmup_steps": warmup_steps, "optimizer": "AdamW", "scheduler": "LinearLR + CosineAnnealingLR", "grad_clip": 1.0},
)

wandb.watch(model, log="all", log_freq=10)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mrudhuhas/.netrc.
wandb: Currently logged in as: mrudhuhas (personal123456) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [17]:


def train_one_epoch(model, dataloader, optimizer, criterion, device, epoch):
    model.train()
    total_loss = 0
    log_interval = 100
    for step, batch in enumerate(dataloader):
        input_ids, target_ids = batch
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        t0 = time.perf_counter()

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(input_ids)
            loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))
        
        loss.backward()
        grad_norm = torch.nn.utils.get_total_norm([p for p in model.parameters() if p.grad is not None])
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        tokens_per_step = input_ids.numel()
        t1 = time.perf_counter()
        dt = t1 - t0

        tokens_per_sec = tokens_per_step / dt

        wandb.log({
            "train/loss": loss.item(),
            "train/learning_rate": optimizer.param_groups[0]['lr'],
            "train/perplexity": torch.exp(loss).item(),
            "train/grad_norm": grad_norm,
            "perf/tokens_per_sec": tokens_per_sec,
            "perf/ms_per_step": dt * 1000,
            "step": epoch * len(dataloader) + step
        })

        if step % log_interval == 0 and step > 0:
            print(f"Epoch [{epoch+1}/{config.num_epochs}] "
                  f"Step [{step}/{len(dataloader)}] "
                  f"Loss: {loss.item():.4f} "
                  f"Perplexity: {torch.exp(loss).item():.4f} "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e} "
                  f"Grad Norm: {grad_norm:.2f} "
                  f"Tokens/sec: {tokens_per_sec:.2f} "
                  f"ms/step: {dt * 1000:.2f}")
    
    avg_loss = total_loss / len(dataloader)
    perplexity = torch.exp(torch.tensor(avg_loss))
    return avg_loss, perplexity.item()

@torch.inference_mode()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    for batch in dataloader:
        input_ids, target_ids = batch
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = model(input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    perplexity = torch.exp(torch.tensor(avg_loss))
    return avg_loss, perplexity.item()

In [18]:
num_epochs = 3
best_valid_loss = float('inf')

for epoch in range(num_epochs):
    train_loss, train_perplexity = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
    val_loss, val_perplexity = evaluate(model, test_loader, criterion, device)

    wandb.log({
        "epoch": epoch + 1,
        "epoch/train_loss": train_loss,
        "epoch/train_perplexity": train_perplexity,
        "epoch/val_loss": val_loss,
        "epoch/val_perplexity": val_perplexity,
    })

    if val_loss < best_valid_loss:
        best_valid_loss = val_loss
        torch.save(model.state_dict(), f"best_model_epoch_{epoch+1}.pth")

    print(f"[Epoch {epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f} - Train Perplexity: {train_perplexity:.4f} - Val Loss: {val_loss:.4f} - Val Perplexity: {val_perplexity:.4f}")


wandb.finish()

Epoch [1/3] Step [100/10792] Loss: 7.6066 Perplexity: 2011.5134 LR: 1.56e-05 Grad Norm: 121.63 Tokens/sec: 62338.05 ms/step: 262.83
Epoch [1/3] Step [200/10792] Loss: 6.0077 Perplexity: 406.5563 LR: 3.11e-05 Grad Norm: 121.87 Tokens/sec: 62335.88 ms/step: 262.83
Epoch [1/3] Step [300/10792] Loss: 4.9130 Perplexity: 136.0481 LR: 4.65e-05 Grad Norm: 122.47 Tokens/sec: 62669.81 ms/step: 261.43
Epoch [1/3] Step [400/10792] Loss: 4.4298 Perplexity: 83.9169 LR: 6.20e-05 Grad Norm: 123.17 Tokens/sec: 61563.89 ms/step: 266.13
Epoch [1/3] Step [500/10792] Loss: 4.1851 Perplexity: 65.7015 LR: 7.74e-05 Grad Norm: 124.03 Tokens/sec: 62431.74 ms/step: 262.43


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f25ed00e7b0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f25ec3dd3b0, execution_count=18 error_before_exec= error_in_exec=None info=<ExecutionInfo object at 7f25ec3dd310, raw_cell="num_epochs = 3
best_valid_loss = float('inf')

for.." transformed_cell="num_epochs = 3
best_valid_loss = float('inf')

for.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/mrudhuhas/Documents/Projects/llm-scratch/story-llm/notebooks/01-test.ipynb#X31sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: [Errno 104] Connection reset by peer